# Bias Audit - Hiring Algorithm (UCI Adult Income)

AAI-540 Final Project supplement. Runs end-to-end on Google Colab free tier.

This notebook trains Logistic Regression, Random Forest, and XGBoost on the UCI
Adult Income dataset and audits each for bias against protected groups
(sex, race, age). Outputs the per-group performance tables, Disparate Impact,
Statistical Parity Difference, Equal Opportunity Difference, and SHAP feature
importance used in the project paper and demo video.

No AWS or SageMaker required.

## 1. Environment setup

In [ ]:
!pip install -q fairlearn shap xgboost scikit-learn==1.3.2 matplotlib pandas numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix
)
import xgboost as xgb
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("Environment ready.")

## 2. Load UCI Adult Income data

In [ ]:
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income"
]

train_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
test_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

train_df = pd.read_csv(train_url, names=COLUMNS, sep=r",\s*", engine="python", na_values="?")
test_df = pd.read_csv(test_url, names=COLUMNS, sep=r",\s*", engine="python", na_values="?", skiprows=1)
test_df["income"] = test_df["income"].str.replace(".", "", regex=False)

raw_df = pd.concat([train_df, test_df], ignore_index=True).dropna().reset_index(drop=True)
raw_df["income_binary"] = (raw_df["income"] == ">50K").astype(int)

print(f"Total rows after dropping NA: {len(raw_df):,}")
print(f"Positive class (>50K) rate: {raw_df['income_binary'].mean():.3f}")
raw_df.head()

## 3. Pre-training bias check (representation in the labels)

In [ ]:
def positive_rate_by_group(df, group_col, label_col="income_binary"):
    return df.groupby(group_col)[label_col].agg(["count", "mean"]).rename(
        columns={"count": "n", "mean": "positive_rate"}
    ).sort_values("positive_rate", ascending=False)

print(">>> Positive rate by SEX")
print(positive_rate_by_group(raw_df, "sex"))
print()
print(">>> Positive rate by RACE")
print(positive_rate_by_group(raw_df, "race"))
print()
age_median = raw_df["age"].median()
raw_df["age_group"] = np.where(raw_df["age"] >= age_median, "older", "younger")
print(f">>> Positive rate by AGE (median split at {age_median:.0f})")
print(positive_rate_by_group(raw_df, "age_group"))

In [ ]:
# Visualize the imbalance in the labels themselves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["sex", "race", "age_group"],
                          ["Sex", "Race", "Age (median split)"]):
    rates = raw_df.groupby(col)["income_binary"].mean().sort_values(ascending=False)
    rates.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Positive rate by {title}")
    ax.set_ylabel("P(income > $50K)")
    ax.set_ylim(0, 0.5)
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("pretraining_bias.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: pretraining_bias.png")

**Takeaway:** The ground-truth labels themselves are imbalanced across protected
groups. Any model trained on this data will inherit that imbalance unless
explicit mitigation is applied. The audit below quantifies how much each model
amplifies, preserves, or reduces that gap.

## 4. Train Logistic Regression, Random Forest, and XGBoost

In [ ]:
NUMERIC = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
CATEGORICAL = ["workclass", "education", "marital_status", "occupation",
               "relationship", "race", "sex", "native_country"]
TARGET = "income_binary"

X = raw_df[NUMERIC + CATEGORICAL]
y = raw_df[TARGET]
protected = raw_df[["sex", "race", "age", "age_group"]].copy()

# Stratified split keeps class balance consistent across train and test
X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X, y, protected, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train):,}   Test: {len(X_test):,}")

In [ ]:
preprocessor = ColumnTransformer([
    ("num", StandardScaler(), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CATEGORICAL),
])

models = {
    "LogisticRegression": Pipeline([
        ("prep", preprocessor),
        ("clf", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "RandomForest": Pipeline([
        ("prep", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=200, max_depth=15,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ]),
    "XGBoost": Pipeline([
        ("prep", preprocessor),
        ("clf", xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                  eval_metric="auc", random_state=RANDOM_STATE,
                                  use_label_encoder=False)),
    ]),
}

predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    predictions[name] = {"y_pred": y_pred, "y_proba": y_proba}
    print(f"{name:20s} trained. AUC = {roc_auc_score(y_test, y_proba):.4f}   "
          f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")

## 5. Overall performance

In [ ]:
def overall_metrics(y_true, y_pred, y_proba):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_proba),
    }

perf_rows = []
for name, p in predictions.items():
    row = overall_metrics(y_test, p["y_pred"], p["y_proba"])
    row["model"] = name
    perf_rows.append(row)

performance_df = pd.DataFrame(perf_rows).set_index("model")[
    ["accuracy", "precision", "recall", "f1", "auc"]
]
performance_df.to_csv("overall_performance.csv")
performance_df.round(4)

## 6. Per-group performance breakdown

In [ ]:
def per_group_performance(y_true, y_pred, group):
    rows = []
    for g, idx in group.groupby(group).groups.items():
        if len(idx) < 30:
            continue
        rows.append({
            "group": g,
            "n": len(idx),
            "accuracy": accuracy_score(y_true.loc[idx], pd.Series(y_pred, index=y_true.index).loc[idx]),
            "selection_rate": pd.Series(y_pred, index=y_true.index).loc[idx].mean(),
            "true_positive_rate": (
                ((pd.Series(y_pred, index=y_true.index).loc[idx] == 1) &
                 (y_true.loc[idx] == 1)).sum() /
                max((y_true.loc[idx] == 1).sum(), 1)
            ),
        })
    return pd.DataFrame(rows).sort_values("selection_rate", ascending=False)

print(">>> XGBoost performance by SEX")
print(per_group_performance(y_test, predictions["XGBoost"]["y_pred"], prot_test["sex"]).round(4))
print()
print(">>> XGBoost performance by RACE")
print(per_group_performance(y_test, predictions["XGBoost"]["y_pred"], prot_test["race"]).round(4))
print()
print(">>> XGBoost performance by AGE GROUP")
print(per_group_performance(y_test, predictions["XGBoost"]["y_pred"], prot_test["age_group"]).round(4))

## 7. Fairness metrics

Three classic group-fairness metrics, computed per protected attribute and per model.

- **Disparate Impact (DI):** P(predict=1 | unprivileged) / P(predict=1 | privileged).
  EEOC "four-fifths rule" treats DI < 0.80 as evidence of adverse impact.
- **Statistical Parity Difference (SPD):** P(predict=1 | unprivileged) - P(predict=1 | privileged).
  Closer to 0 is more parity.
- **Equal Opportunity Difference (EOD):** TPR(unprivileged) - TPR(privileged).
  Closer to 0 means qualified people are recommended at the same rate across groups.

In [ ]:
def fairness_metrics(y_true, y_pred, group_series, privileged_value):
    y_true = pd.Series(y_true).reset_index(drop=True)
    y_pred = pd.Series(y_pred).reset_index(drop=True)
    group_series = pd.Series(group_series).reset_index(drop=True)

    priv = group_series == privileged_value
    unpriv = ~priv

    p_priv = y_pred[priv].mean()
    p_unpriv = y_pred[unpriv].mean()

    di = p_unpriv / p_priv if p_priv > 0 else np.nan
    spd = p_unpriv - p_priv

    tpr_priv = ((y_pred[priv] == 1) & (y_true[priv] == 1)).sum() / max((y_true[priv] == 1).sum(), 1)
    tpr_unpriv = ((y_pred[unpriv] == 1) & (y_true[unpriv] == 1)).sum() / max((y_true[unpriv] == 1).sum(), 1)
    eod = tpr_unpriv - tpr_priv

    return {"disparate_impact": di, "statistical_parity_diff": spd, "equal_opportunity_diff": eod}


fairness_rows = []
for model_name, p in predictions.items():
    for attr, priv_value in [("sex", "Male"), ("race", "White"), ("age_group", "older")]:
        m = fairness_metrics(y_test.values, p["y_pred"], prot_test[attr].values, priv_value)
        m["model"] = model_name
        m["protected_attr"] = attr
        m["privileged_group"] = priv_value
        fairness_rows.append(m)

fairness_df = pd.DataFrame(fairness_rows)[
    ["model", "protected_attr", "privileged_group",
     "disparate_impact", "statistical_parity_diff", "equal_opportunity_diff"]
]
fairness_df.to_csv("fairness_metrics.csv", index=False)
fairness_df.round(4)

In [ ]:
# Visualize Disparate Impact across models and groups
fig, ax = plt.subplots(figsize=(11, 5))
pivot = fairness_df.pivot(index="protected_attr", columns="model", values="disparate_impact")
pivot.plot(kind="bar", ax=ax, edgecolor="black")
ax.axhline(0.80, color="red", linestyle="--", label="Four-fifths rule (0.80)")
ax.axhline(1.25, color="red", linestyle="--", alpha=0.5)
ax.axhline(1.00, color="green", linestyle="-", alpha=0.3, label="Parity (1.0)")
ax.set_ylabel("Disparate Impact")
ax.set_title("Disparate Impact by model and protected attribute")
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
ax.legend()
plt.tight_layout()
plt.savefig("disparate_impact.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: disparate_impact.png")

## 8. SHAP explainability (what features drive the predictions)

In [ ]:
# Use XGBoost for SHAP because TreeExplainer is fast and exact
xgb_pipeline = models["XGBoost"]
xgb_clf = xgb_pipeline.named_steps["clf"]
prep = xgb_pipeline.named_steps["prep"]

X_test_transformed = prep.transform(X_test)
feature_names = (NUMERIC +
                 list(prep.named_transformers_["cat"].get_feature_names_out(CATEGORICAL)))

explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test_transformed[:2000])  # sample to keep it fast

shap.summary_plot(shap_values, X_test_transformed[:2000],
                  feature_names=feature_names, max_display=15, show=False)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved: shap_summary.png")

In [ ]:
# Mean absolute SHAP value per feature - which features matter most overall
shap_importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

# Flag features that ARE protected attributes or proxies
def flag_protected(name):
    for p in ["sex_", "race_", "marital_status_", "relationship_"]:
        if name.startswith(p):
            return "protected_or_proxy"
    return "neutral"

shap_importance["flag"] = shap_importance["feature"].apply(flag_protected)
shap_importance.head(20)

**Key observation:** If features starting with `sex_`, `race_`, `marital_status_`,
or `relationship_` appear high in the SHAP ranking, the model is leaning on
protected attributes (or proxies for them) to make decisions. That is the
mechanism behind the disparate impact numbers above. The paper's discussion
section should reference these features directly.

## 9. Summary table for the paper

In [ ]:
print("=" * 70)
print("OVERALL PERFORMANCE")
print("=" * 70)
print(performance_df.round(4))
print()
print("=" * 70)
print("FAIRNESS AUDIT (DI, SPD, EOD by model and protected group)")
print("=" * 70)
print(fairness_df.round(4).to_string(index=False))
print()
print("=" * 70)
print("ARTIFACTS SAVED FOR PAPER AND DEMO")
print("=" * 70)
print(" - pretraining_bias.png")
print(" - disparate_impact.png")
print(" - shap_summary.png")
print(" - overall_performance.csv")
print(" - fairness_metrics.csv")

## 10. Recommended mitigation (paper Discussion section)

Three options ordered by how disruptive they are to the current pipeline:

1. **Threshold tuning per group** — keep the model, pick a different decision
   threshold for the unprivileged group so selection rates match. Cheap, fast,
   reversible. Hurts overall accuracy slightly.
2. **Reweighing the training data** — give underrepresented positive examples
   higher sample weights during training. Implemented in one line with
   `fairlearn.preprocessing.CorrelationRemover` or `aif360.algorithms.preprocessing.Reweighing`.
3. **Adversarial debiasing** — train a second network that tries to predict the
   protected attribute from the main model's predictions, and penalize the
   main model when that adversary succeeds. Most powerful, most complex.

In a hiring context the right answer is almost never "deploy the model with
no mitigation." The four-fifths rule is enforceable under EEOC guidance, so a
DI below 0.80 on any protected attribute is a compliance risk in addition
to an ethical one.